# VoiceDesignCloner on Google Colab

このNotebookは、現在の作業ツリーから作ったソースZIPをColabへ取り込み、GPU上でIrodori-TTS V4対応のVoiceDesignClonerを起動します。起動時にGradioの一時公開URLを発行します。

**対象:** Windows上のこのツールをColab GPUで使いたい方  
**必要なもの:** Google Colab、GPUランタイム、ローカルで生成したソースZIP

## 手順

1. ColabのランタイムをGPUにする
2. 作業ツリーから作ったソースZIPをアップロードする
3. 出力先としてGoogle Driveをマウントする
4. 依存パッケージとIrodori-TTSをセットアップする
5. アプリを起動し、セル出力の公開URLを開く

> Gradioの共有URLは公開リンクです。URLを知っている人はアプリにアクセスできます。Colabランタイムを停止するとリンクは無効になります。恒久的なホスティングURLではありません。

[Gradio共有リンクの説明](https://www.gradio.app/guides/understanding-gradio-share-links) · [Colabのリソース制限とGPU提供について](https://research.google.com/colaboratory/faq.html)

## 1. GPUランタイムを選ぶ

Colabメニューの **ランタイム → ランタイムのタイプを変更 → ハードウェア アクセラレータ: GPU** を選びます。GPUの種類と利用枠はColab側で変動します。Irodori V4 Fullには、利用可能ならL4またはA100などのAmpere世代以降をおすすめします。

In [ ]:
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPUが見つかりません。ランタイムのタイプをGPUに変更してから、このセルを再実行してください。")

gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
print(f"Python: {sys.version.split()[0]}")
print(f"GPU: {gpu_name} | compute capability: {capability[0]}.{capability[1]}")
print(f"VRAM: {props.total_memory / 1024**3:.1f} GiB")
if capability[0] < 8:
    print("このGPUはネイティブBF16世代ではありません。V4タブの精度欄でFP32を手動選択してください。")
else:
    print("V4 FullではBF16を使用できます。")

## 2. 現在の作業ツリーをZIPにする

ColabからGitHubをcloneすると、未コミットのV4変更が含まれないことがあります。Windows側のリポジトリ直下で次を実行し、今のソースをまとめてください。

```powershell
python .\colab\package_colab_source.py --force
```

作成先は `output/colab/voice_design_cloner_colab_source.zip` です。このZIPにはアプリのソース、内蔵コーパス、プリセットを含めます。音声出力、参照音声、LoRA、設定ファイル、ログ、仮想環境、Git履歴は含めません。

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith(".zip")]
if len(archives) != 1:
    raise RuntimeError(f"ソースZIPを1個だけ選択してください。選択されたZIP: {archives}")

SOURCE_ZIP = Path("/content") / archives[0]
print(f"受け取りました: {SOURCE_ZIP.name} ({SOURCE_ZIP.stat().st_size / 1024**2:.2f} MiB)")

In [ ]:
import hashlib
import json
import zipfile
from pathlib import Path, PurePosixPath

WORKSPACE = Path("/content/voice-design-cloner")
if WORKSPACE.exists():
    raise RuntimeError("作業フォルダが既にあります。新しいColabランタイムで最初から実行してください。")

with zipfile.ZipFile(SOURCE_ZIP) as archive:
    members = archive.namelist()
    manifest_name = "colab_source_manifest.json"
    if manifest_name not in members:
        raise RuntimeError("Colab用ソースマニフェストがありません。ローカルでZIPを作り直してください。")
    manifest = json.loads(archive.read(manifest_name))
    if manifest.get("format") != "voice-design-cloner-colab-source":
        raise RuntimeError("別のZIPが選択されています。VoiceDesignCloner用のZIPを選んでください。")
    for name in members:
        member = PurePosixPath(name)
        if member.is_absolute() or ".." in member.parts:
            raise RuntimeError(f"安全でないZIP内パス: {name}")
    for item in manifest["files"]:
        payload = archive.read(item["path"])
        if len(payload) != item["size_bytes"] or hashlib.sha256(payload).hexdigest() != item["sha256"]:
            raise RuntimeError(f"ZIP内ファイルの検証に失敗しました: {item['path']}")
    archive.extractall(WORKSPACE)

if not (WORKSPACE / "app.py").is_file() or not (WORKSPACE / "ui/tab_irodori_v4.py").is_file():
    raise RuntimeError("必要なアプリファイルがありません。最新のソースZIPを作ってください。")
print(f"展開完了: {WORKSPACE}")
print(f"検証済みファイル数: {len(manifest['files'])}")

## 3. 生成音声の保存先を用意する

Driveをマウントすると、Colabを切断したあとも生成結果とLoRAを残せます。出力先をDriveにしない場合は、次のセルの `PERSIST_OUTPUTS_TO_DRIVE` を `False` にしてください。モデル本体とPython環境はColab側に置かれ、ランタイムをリセットすると再セットアップが必要です。

既存の参照音声やLoRAはソースZIPに含めていません。Driveマウント後、必要なら `MyDrive/VoiceDesignCloner/output/voice_design/` または `MyDrive/VoiceDesignCloner/output/lora_v4/` に置いてください。

In [ ]:
from pathlib import Path

PERSIST_OUTPUTS_TO_DRIVE = True
LOCAL_OUTPUT = WORKSPACE / "output"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_APP_DIR = Path("/content/drive/MyDrive/VoiceDesignCloner")
    DRIVE_OUTPUT = DRIVE_APP_DIR / "output"
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
    if LOCAL_OUTPUT.exists() or LOCAL_OUTPUT.is_symlink():
        raise RuntimeError(f"出力フォルダが既にあります。上書きを避けるため停止しました: {LOCAL_OUTPUT}")
    LOCAL_OUTPUT.symlink_to(DRIVE_OUTPUT, target_is_directory=True)
    APP_OUTPUT = DRIVE_OUTPUT
else:
    LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
    APP_OUTPUT = LOCAL_OUTPUT

for folder in ("voice_design", "voice_clone", "lora", "lora_data", "lora_v4", "lora_data_v4", "irodori_infer"):
    (APP_OUTPUT / folder).mkdir(parents=True, exist_ok=True)
print(f"生成結果の保存先: {APP_OUTPUT}")

## 4. 依存関係とモデルをセットアップする

セットアップには時間と数GBのダウンロードが必要です。Colab上でGPUを確認したあと、既存のLinuxセットアップスクリプトを使います。PyTorch CUDA 12.8を指定し、Irodori-TTSは専用仮想環境に分けてインストールします。

In [ ]:
import os
import subprocess

install_env = os.environ.copy()
install_env["VDC_TORCH_CUDA"] = "cu128"
result = subprocess.run(["bash", "setup.sh"], cwd=WORKSPACE, env=install_env)
if result.returncode != 0:
    raise RuntimeError(f"セットアップが終了コード {result.returncode} で失敗しました。上のログを確認してください。")

IRODORI_PYTHON = Path.home() / ".vdc-engines" / "Irodori-TTS" / ".venv" / "bin" / "python"
if not IRODORI_PYTHON.is_file():
    raise RuntimeError(f"Irodori-TTS環境が見つかりません: {IRODORI_PYTHON}")
print(f"アプリ環境: {WORKSPACE / 'venv' / 'bin' / 'python'}")
print(f"Irodori環境: {IRODORI_PYTHON}")

## 5. アプリを起動して共有URLを取得する

このセルはアプリが動いている間実行中のままです。出力ログに表示される `https://....gradio.live` のURLを開いてください。URLを知っている人はアクセスできます。使用後はセルの停止ボタンを押してください。

In [ ]:
import os
import signal
import subprocess

launch_env = os.environ.copy()
launch_env["VDC_SERVER_NAME"] = "0.0.0.0"
launch_env["VDC_SERVER_PORT"] = "7860"
launch_env["VDC_SHARE"] = "1"
launch_env["VDC_RESTART"] = "1"
app_python = WORKSPACE / "venv" / "bin" / "python"

process = subprocess.Popen(
    [str(app_python), "-u", "app.py"],
    cwd=WORKSPACE,
    env=launch_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
try:
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f"VoiceDesignClonerが終了コード {return_code} で終了しました。")
except KeyboardInterrupt:
    print("\nアプリを停止しています…", flush=True)
    process.send_signal(signal.SIGINT)
    try:
        process.wait(timeout=20)
    except subprocess.TimeoutExpired:
        process.terminate()
        process.wait(timeout=5)
finally:
    if process.poll() is None:
        process.kill()
        process.wait()
print("アプリを停止しました。共有URLは利用できなくなりました。")